# {{TITLE}}

ESP database exploration — {{DATE_LONG}}

Use this notebook to explore the ESP PostgreSQL database, especially the EAV
(Entity-Attribute-Value) schema. Pre-loaded with common query patterns and
visualization setup.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.home() / '.claude/skills/jupyter-notebook/lib'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

try:
    import plotly.express as px
    import plotly.graph_objects as go
    HAS_PLOTLY = True
except ImportError:
    HAS_PLOTLY = False

from jupyter_utils import get_db_engine, query_df, setup_plotting

plt, sns = setup_plotting()
engine = get_db_engine()
print('Connected to ESP database')

## Schema Overview

Quick look at table sizes and the EAV structure.

In [ ]:
# Table row counts (top 20 by size)
table_sizes = query_df("""
    SELECT schemaname, relname AS table_name,
           n_live_tup AS row_count
    FROM pg_stat_user_tables
    ORDER BY n_live_tup DESC
    LIMIT 20
""")
table_sizes

In [ ]:
if not table_sizes.empty:
    fig, ax = plt.subplots(figsize=(14, 8))
    table_sizes.plot(
        x='table_name', y='row_count', kind='barh',
        ax=ax, color='steelblue', legend=False
    )
    ax.set_title('Top 20 Tables by Row Count', fontweight='bold')
    ax.set_xlabel('Rows')
    ax.set_ylabel('')
    ax.invert_yaxis()
    for container in ax.containers:
        ax.bar_label(container, fmt='%,.0f', padding=5)
    plt.tight_layout()
    plt.show()

## EAV Exploration

Common patterns for querying the Entity-Attribute-Value tables.

In [ ]:
# Entity types and counts
entity_types = query_df("""
    SELECT cd.name AS entity_type, COUNT(*) AS count
    FROM resource r
    JOIN esp_cls_definition cd ON r.cls = cd.clsid
    GROUP BY cd.name
    ORDER BY count DESC
""")
entity_types['count'] = entity_types['count'].astype(int)
entity_types

In [ ]:
if not entity_types.empty and HAS_PLOTLY:
    fig = px.pie(entity_types, values='count', names='entity_type',
                 title='Entity Type Distribution')
    fig.show()
elif not entity_types.empty:
    fig, ax = plt.subplots()
    entity_types.set_index('entity_type')['count'].plot(
        kind='pie', ax=ax, autopct='%1.1f%%'
    )
    ax.set_title('Entity Type Distribution')
    ax.set_ylabel('')
    plt.tight_layout()
    plt.show()

## Custom Queries

Add your exploration queries below.

In [ ]:
# Example: query a specific table
# df = query_df("SELECT * FROM {{TABLE}} LIMIT 100")
# df.head()

## Findings

_Record observations here._